# Notebook 2 — Chunking y Formateo de splits

Objetivo: cargar los splits previamente construidos, chunkearlos y transformarlos en **dos formatos de entrada/salida distintos**, uno por familia de arquitectura:

- **Encoder-only (BERT / clinical-BERT):** clasificación de tokens, esquema BIO.
- **Encoder-decoder (mT5):** texto → texto, generando el **nombre** de la enfermedad normalizada
  (no el código SNOMED).

Ambos formatos parten del **mismo split** train/dev/test para que la comparación entre modelos
en el Notebook 2 sea justa.

## 1. Setup e imports

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")

except ImportError:
    SPLITS_DIR = "."

In [ ]:
# !pip install datasets huggingface_hub --quiet

import json
import os
import random
import numpy as np
from pathlib import Path
from collections import Counter

from datasets import Dataset, DatasetDict

random.seed(42)

SPLITS_DIR = ("/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M1/distemist_final")

In [ ]:
# --- Cargar splits crudos de Notebook 1 -------------------------------
from datasets import load_from_disk

splits_dir = Path(SPLITS_DIR)
raw_splits = load_from_disk(str(splits_dir / "distemist_raw_splits"))

print(raw_splits)
print(raw_splits["train"][0])  # confirma que llegan doc_id, text, entities intactos

## 2. Tokenizadores

En esta sección se decidirá la máxima cantidad de palabras por muestra para hacer chunking a partir de esta.

In [ ]:
# Diagnóstico: qué tenemos ANTES de tocar nada
import importlib.metadata as md_, sys

print('python', sys.version.split()[0])
for p in ['torch', 'transformers', 'tokenizers', 'sentencepiece', 'pandas']:
    try:
        print(f'{p:16} {md_.version(p)}')
    except md_.PackageNotFoundError:
        print(f'{p:16} FALTA')

In [ ]:
try:
    import transformers, sentencepiece  # noqa: F401
    print('Entorno completo — no instalo nada.')
except ImportError:
    %pip -q install transformers sentencepiece
    print('Instalado. Si Colab pide RESTART RUNTIME, reinicia y vuelve a correr desde aquí.')

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer

torch.manual_seed(42)
pd.set_option('display.max_colwidth', None)
print('torch', torch.__version__, '| listo')

In [ ]:
# En TOKENIZADORES incluir aquellos que se quieran probar para cada arquitectura específica.
TOKENIZADORES = {
    'BERT-en':'bert-base-uncased',
    'BERT-clinico-en': 'medicalai/ClinicalBERT',
    'Roberta-clinico-es': 'BSC-TeMU/roberta-base-biomedical-es',
    'mT5': 'google/mt5-small',
}

toks = {}
for nombre, ruta in TOKENIZADORES.items():
    toks[nombre] = AutoTokenizer.from_pretrained(ruta)
    print(f'{nombre:24} vocabulario: {toks[nombre].vocab_size:>7,} tokens')

Visualizemos cómo cada tokenizador despedaza el mismo texto.

In [ ]:
def comparar(texto):
    """Muestra cómo cada tokenizador despedaza el mismo texto."""
    filas = []
    for nombre, tk in toks.items():
        piezas = tk.tokenize(texto)
        filas.append({'Tokenizador': nombre, 'N': len(piezas), 'Tokens': ' | '.join(piezas)})
    return pd.DataFrame(filas).set_index('Tokenizador')

comparar('Varón de 35 años, con antecedentes de lúes, hepatitis no filiada y circuncisión a los 4 años, que consulta por bifidez del chorro miccional.')

A continuación calculamos estadísticas de longitud de secuencias  tokens para cada tokenizador.

In [ ]:
def measure_tokens(records, tokenizer):
    lengths = []
    for rec in records:
        lengths.append(len(tokenizer.encode(rec["text"], add_special_tokens=True)))
    return lengths


def calculate_lengths(toks, split, split_name="", tokenizer_name=""):
    """Calcula estadísticas de longitud de secuencias de tokens."""

    if tokenizer_name:
        length = measure_tokens(split, toks[tokenizer_name])
        print(f"{tokenizer_name} ({split_name}):")
        print("Máximo:", max(length))
        print("Media:", np.mean(length))
        print("Mediana:", np.median(length))
        print("P70:", np.percentile(length, 70))
        print("P99:", np.percentile(length, 99))
        print('\n')

    else:
      for tokenizer_name, tokenizer in toks.items():
          lengths = measure_tokens(split, tokenizer)
          print(f"{tokenizer_name} ({split_name}):")
          print("Máximo:", max(lengths))
          print("Media:", np.mean(lengths))
          print("Mediana:", np.median(lengths))
          print("P70:", np.percentile(lengths, 70))
          print("P99:", np.percentile(lengths, 99))
          print('\n')

In [ ]:
for split_name, split_data in raw_splits.items():
    calculate_lengths(toks, split_data, split_name=split_name)

Ahora exploramos el ratio de tokens/palabras para cada tokenizador. Este es un punto importante para decidir el tokenizador y el máximo de palabras para el chunking.

In [ ]:
def compute_safe_window_words(tokenizer_name, tokenizer, texts, target_max_tokens=480, safety_margin=0.9):
    """
    target_max_tokens: tope real que quieres respetar (dejamos margen bajo 512
                        para [CLS]/[SEP] y para no quedar justo en el límite).
    safety_margin: reduce el resultado un poco más, para cubrir documentos
                   con vocabulario más fragmentado que el promedio de la muestra.
    """
    ratios = []
    for text in texts:
        n_words = len(__import__("re").findall(r"\S+", text))
        n_tokens = len(tokenizer.encode(text, add_special_tokens=False))
        if n_words > 0:
            ratios.append(n_tokens / n_words)

    # usamos el percentil 90 de fragmentación, no el promedio, para no
    # subestimar documentos con vocabulario más raro/fragmentado que el típico
    worst_case_ratio = np.percentile(ratios, 100)
    safe_window = int((target_max_tokens / worst_case_ratio) * safety_margin)

    print("\nTokenizador: ", tokenizer_name)
    print(f"Ratio tokens/palabra (p90): {worst_case_ratio:.2f}")
    print(f"window_words seguro para este tokenizer: {safe_window}")
    return safe_window

In [ ]:
sample_texts = [
    text
    for split_data in raw_splits.values()
    for text in split_data["text"]
]

print(f"Total de documentos: {len(sample_texts)}")

windows = {}

for tokenizer_name, tokenizer in toks.items():
  windows[tokenizer_name] = compute_safe_window_words(tokenizer_name, tokenizer, sample_texts)

---

### **PUNTO DECISIVO:** A partir de las visualizaciones anteriores y las cifras halladas tomar 2 decisiones:


1. Tokenizador elegido.
2. Máximo de palabras ó ventana de palabras por documento (más abajo esta cifra se define como `window_words`).


In [ ]:
# Definir tokenizador elegido
chosen_tokenizer_name = 'Roberta-clinico-es'

# Definir tamaño de ventana de palabras
chosen_window_words = 277

## 5. Función de Chunking

Con esta función se resuelve el problema hallado en el N1 (secuencias de tokens de longitud mayor a 512) a partir de chunking con solapamiento para no perder las entidades en nuestros textos y tener el tamaño adecuado. Usaremos el umbral de palabras elegido, no de tokens.

In [ ]:
def chunk_record_by_words(rec, window_words=chosen_window_words, overlap_words=50):
    text = rec["text"]
    word_matches = list(__import__("re").finditer(r"\S+", text))

    if len(word_matches) <= window_words:
        return [rec]  # no necesita chunking

    chunks = []
    start_idx = 0
    chunk_id = 0
    while start_idx < len(word_matches):
        end_idx = min(start_idx + window_words, len(word_matches))
        char_start = word_matches[start_idx].start()
        char_end = word_matches[end_idx - 1].end()

        chunk_entities = [
            {**ent, "start": ent["start"] - char_start, "end": ent["end"] - char_start}
            for ent in rec["entities"]
            if ent["start"] >= char_start and ent["end"] <= char_end
        ]

        chunks.append({
            "doc_id": f"{rec['doc_id']}_chunk{chunk_id}",
            "text": text[char_start:char_end],
            "entities": chunk_entities,
        })

        if end_idx == len(word_matches):
            break
        start_idx = end_idx - overlap_words  # overlap para no cortar entidades en el borde
        chunk_id += 1

    return chunks


def chunk_all_records(records, window_words=chosen_window_words, overlap_words=50):
    chunked = []
    for rec in records:
        chunked.extend(chunk_record_by_words(rec, chosen_window_words, overlap_words))
    return chunked

In [ ]:
chunked_splits = {name: chunk_all_records(recs) for name, recs in raw_splits.items()}

Ahora confirmamos que las secuencias de tokens no excedan el límite del modelo para la siguiente etapa.

In [ ]:
for split_name, split_data in raw_splits.items():
    calculate_lengths(toks, split_data, split_name=split_name, tokenizer_name=chosen_tokenizer_name)

## 4. Formato para BERT / clinical-BERT (token classification, BIO)

Convertimos cada documento en tokens + etiquetas BIO. Aquí usamos un tokenizador *whitespace*
simple para construir las etiquetas alineadas a nivel de carácter → palabra; en el Notebook 2,
el tokenizador real del modelo (WordPiece/BPE) se encarga de re-alinear estas etiquetas a
subpalabras con `tokenize_and_align_labels` (patrón estándar de Hugging Face para NER).


In [ ]:
def char_spans_to_bio(text, entities):
    # Tokenización simple por espacios, con offsets de carácter por token
    tokens, offsets = [], []
    for match in __import__("re").finditer(r"\S+", text):
        tokens.append(match.group())
        offsets.append((match.start(), match.end()))

    tags = ["O"] * len(tokens)
    for ent in entities:
        first_token = True
        for i, (tok_start, tok_end) in enumerate(offsets):
            if tok_end <= ent["start"] or tok_start >= ent["end"]:
                continue
            tags[i] = "B-ENFERMEDAD" if first_token else "I-ENFERMEDAD"
            first_token = False
    return tokens, tags


def build_bert_split(records):
    data = {"doc_id": [], "tokens": [], "ner_tags": []}
    for rec in records:
        tokens, tags = char_spans_to_bio(rec["text"], rec["entities"])
        data["doc_id"].append(rec["doc_id"])
        data["tokens"].append(tokens)
        data["ner_tags"].append(tags)
    return Dataset.from_dict(data)


In [ ]:
bert_dataset = DatasetDict({
    split_name: build_bert_split(records) for split_name, records in chunked_splits.items()
})
print(bert_dataset)
print("\nEjemplo de formato:\n")
print(bert_dataset["train"][0])

## 5. Formato para mT5 (texto → texto)

Input: el texto clínico completo (o una ventana de contexto si el documento es muy largo).
Output: una lista estructurada de entidades usando el **término normalizado en español**
(no el código SNOMED) — así aprovechamos el conocimiento léxico-médico que mT5 ya trae del
preentrenamiento, en vez de pedirle que memorice un string opaco.

Formato de salida elegido (parseable de forma determinística en evaluación):
`entidad1 [SEP] entidad2 [SEP] ...`


In [ ]:
ENTITY_SEP = " [SEP] "

def build_mt5_target(entities):
    terms = [ent.get("normalized_term", ent["text"]) for ent in entities]
    # sin duplicados, orden estable
    seen = []
    for t in terms:
        if t not in seen:
            seen.append(t)
    return ENTITY_SEP.join(seen) if seen else "ninguna"


def build_mt5_split(records):
    data = {"doc_id": [], "input_text": [], "target_text": []}
    for rec in records:
        data["doc_id"].append(rec["doc_id"])
        data["input_text"].append(
            "Identifica las enfermedades mencionadas en el siguiente texto clínico: " + rec["text"]
        )
        data["target_text"].append(build_mt5_target(rec["entities"]))
    return Dataset.from_dict(data)

In [ ]:
mt5_dataset = DatasetDict({
    split_name: build_mt5_split(records) for split_name, records in chunked_splits.items()
})
print(mt5_dataset)
print(mt5_dataset["train"][0])

## 6. Guardar datasets procesados

Se guardan para que el Notebook 3 los cargue directamente, sin repetir el
procesamiento y garantizando que los splits de los tres modelos usan **exactamente los mismos ejemplos**.

Nota: A pesar de que la cantidad de documentos por splits puede variar dadas las diferentes condiciones de los tokenizadores (cada uno puede necesitar una ventana de contexto diferente), se mantiene la consistencia para comparar entre modelos ya que los splits siguen estando compuestos de los mismos casos.


In [ ]:
splits_dir = Path(SPLITS_DIR)

bert_dataset.save_to_disk(str(splits_dir / "distemist_bert_format"))
mt5_dataset.save_to_disk(str(splits_dir / "distemist_mt5_format"))

print("Guardado en:", splits_dir.resolve())
